# 2026 NLP Competition Solution

Ноутбук выполняет полный цикл: загрузка данных, EDA, обучение, инференс, постпроцессинг и сохранение `sample_submission.csv`.

In [ ]:
import ast
import platform
import random
import re
import warnings
from dataclasses import dataclass
from html import unescape
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from scipy.sparse import hstack
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.multiclass import OneVsRestClassifier
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
except ImportError:
    MultilabelStratifiedKFold = None

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

SEED = 322
TARGET_COLUMNS = ["label_0", "label_1", "label_2", "label_3", "label_4"]


def detect_data_root() -> Path:
    # Local run first.
    local_root = Path(".")
    if (local_root / "train.csv").exists() and (local_root / "test.csv").exists() and (local_root / "sample_submission.csv").exists():
        return local_root

    # Kaggle competition input (folder name can vary by slug).
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for train_path in kaggle_input.rglob("train.csv"):
            root = train_path.parent
            if (root / "test.csv").exists() and (root / "sample_submission.csv").exists():
                return root

    raise FileNotFoundError("Could not find train.csv, test.csv, sample_submission.csv")


PROJECT_ROOT = detect_data_root()


@dataclass
class DataBundle:
    train: pd.DataFrame
    test: pd.DataFrame
    sample_submission: pd.DataFrame


def set_global_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def parse_target(value: str) -> np.ndarray:
    arr = np.asarray(ast.literal_eval(value), dtype=np.int64)
    if arr.shape != (5,):
        raise ValueError(f"Target must have 5 labels, got {arr.shape}.")
    if not np.isin(arr, [0, 1]).all():
        raise ValueError("Target values must be binary.")
    return arr


def load_competition_data(data_dir: Path | str = ".") -> DataBundle:
    data_dir = Path(data_dir)
    train = pd.read_csv(data_dir / "train.csv", sep="\t")
    test = pd.read_csv(data_dir / "test.csv", sep="\t")
    sample_submission = pd.read_csv(data_dir / "sample_submission.csv")
    return DataBundle(train=train, test=test, sample_submission=sample_submission)


def validate_schema(train: pd.DataFrame, test: pd.DataFrame, sample_submission: pd.DataFrame) -> None:
    if set(train.columns) != {"id", "source", "title", "text", "publication_date", "target"}:
        raise ValueError(f"Unexpected train columns: {train.columns.tolist()}")
    if set(test.columns) != {"id", "source", "title", "text", "publication_date"}:
        raise ValueError(f"Unexpected test columns: {test.columns.tolist()}")
    if set(sample_submission.columns) != {"id", "target"}:
        raise ValueError(f"Unexpected submission columns: {sample_submission.columns.tolist()}")


def add_target_columns(train: pd.DataFrame) -> pd.DataFrame:
    parsed = np.vstack(train["target"].map(parse_target).to_numpy())
    out = train.copy()
    for i, c in enumerate(TARGET_COLUMNS):
        out[c] = parsed[:, i]
    return out


def extract_targets(df: pd.DataFrame) -> np.ndarray:
    return df[TARGET_COLUMNS].to_numpy(dtype=np.int64)


def format_submission_targets(binary_matrix: np.ndarray) -> list[str]:
    return ["[" + ",".join(map(str, row.astype(int).tolist())) + "]" for row in binary_matrix]


def hamming_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(1.0 - np.not_equal(y_true, y_pred).mean())


TAG_RE = re.compile(r"<[^>]+>")
SPACE_RE = re.compile(r"\s+")


def clean_text(value: str) -> str:
    if not isinstance(value, str):
        value = "" if pd.isna(value) else str(value)
    text = unescape(value)
    text = TAG_RE.sub(" ", text)
    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    return SPACE_RE.sub(" ", text).strip()


def compose_text_features(df: pd.DataFrame) -> pd.Series:
    title = df["title"].fillna("").map(clean_text)
    text = df["text"].fillna("").map(clean_text)
    source = df["source"].fillna("").astype(str)
    dt = pd.to_datetime(df["publication_date"], errors="coerce")
    year = dt.dt.year.fillna(-1).astype(int).astype(str)
    month = dt.dt.month.fillna(-1).astype(int).astype(str)
    return "[SRC] " + source + " [YEAR] " + year + " [MONTH] " + month + " [TITLE] " + title + " [TEXT] " + text


def _mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    pooled = (last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
    return pooled


def build_transformer_embeddings(texts: pd.Series, model_name: str, batch_size: int, max_length: int, cache_path: Path | None = None) -> np.ndarray:
    if cache_path is not None and cache_path.exists():
        return np.load(cache_path)["embeddings"]

    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    chunks = []
    with torch.inference_mode():
        for start in tqdm(range(0, len(texts), batch_size), desc=f"Embeddings:{model_name}"):
            batch = [f"passage: {t}" for t in texts.iloc[start:start + batch_size].tolist()]
            encoded = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(device)
            with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
                outputs = model(**encoded)
                pooled = _mean_pool(outputs.last_hidden_state, encoded["attention_mask"])
                pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
            chunks.append(pooled.float().cpu().numpy())

    embeddings = np.vstack(chunks)
    if cache_path is not None:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(cache_path, embeddings=embeddings)
    return embeddings


def build_tfidf_features(train_texts: pd.Series, test_texts: pd.Series, max_features_word: int = 120000, max_features_char: int = 80000):
    word = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.98, sublinear_tf=True, max_features=max_features_word)
    char = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.98, sublinear_tf=True, max_features=max_features_char)
    x_train = hstack([word.fit_transform(train_texts), char.fit_transform(train_texts)]).tocsr()
    x_test = hstack([word.transform(test_texts), char.transform(test_texts)]).tocsr()
    return x_train, x_test


def build_base_classifier(seed: int = SEED, C: float = 2.0):
    base = LogisticRegression(C=C, max_iter=600, solver="liblinear", random_state=seed)
    return OneVsRestClassifier(base)


def make_multilabel_splitter(n_splits: int = 5, seed: int = SEED):
    if MultilabelStratifiedKFold is not None:
        return MultilabelStratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return KFold(n_splits=n_splits, shuffle=True, random_state=seed)


def fit_cv_models(x, y: np.ndarray, estimator, splitter, score_fn):
    oof = np.zeros((y.shape[0], y.shape[1]), dtype=np.float32)
    models, fold_scores = [], []
    for fold_idx, (tr_idx, va_idx) in enumerate(splitter.split(x, y), start=1):
        model = clone(estimator)
        model.fit(x[tr_idx], y[tr_idx])
        proba = model.predict_proba(x[va_idx])
        oof[va_idx] = proba
        pred = (proba >= 0.5).astype(np.int64)
        score = score_fn(y[va_idx], pred)
        fold_scores.append(float(score))
        models.append(model)
        print(f"Fold {fold_idx}: hamming_score={score:.6f}")
    return {"oof_proba": oof, "models": models, "fold_scores": fold_scores}


def fit_full_model(x, y: np.ndarray, estimator):
    model = clone(estimator)
    model.fit(x, y)
    return model


def find_best_thresholds(y_true: np.ndarray, y_proba: np.ndarray, score_fn, grid: np.ndarray | None = None):
    if grid is None:
        grid = np.linspace(0.2, 0.8, 61)
    thresholds = np.full(y_true.shape[1], 0.5, dtype=np.float32)
    best_global = score_fn(y_true, (y_proba >= thresholds).astype(np.int64))
    for label in range(y_true.shape[1]):
        best_t, best_score = thresholds[label], best_global
        for t in grid:
            trial = thresholds.copy()
            trial[label] = float(t)
            score = score_fn(y_true, (y_proba >= trial).astype(np.int64))
            if score > best_score:
                best_t, best_score = float(t), score
        thresholds[label] = best_t
        best_global = best_score
    return thresholds, float(best_global)


def apply_thresholds(y_proba: np.ndarray, thresholds: np.ndarray) -> np.ndarray:
    return (y_proba >= thresholds).astype(np.int64)


set_global_seed(SEED)
print(f"Seed fixed to: {SEED}")
print(f"Python: {platform.python_version()}")

In [ ]:
bundle = load_competition_data(PROJECT_ROOT)
train_df = bundle.train
test_df = bundle.test
sample_submission = bundle.sample_submission

validate_schema(train_df, test_df, sample_submission)
train_df = add_target_columns(train_df)

print("Data root:", PROJECT_ROOT)
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Sample submission shape:", sample_submission.shape)
display(train_df.head(2))

In [ ]:
# EDA
y = extract_targets(train_df)
label_freq = pd.Series(y.mean(axis=0), index=TARGET_COLUMNS, name="positive_ratio")
cardinality = pd.Series(y.sum(axis=1), name="labels_per_sample")

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
label_freq.plot(kind="bar", ax=axes[0], title="Label positive ratio")
cardinality.value_counts().sort_index().plot(kind="bar", ax=axes[1], title="Label cardinality")

tmp = train_df.copy()
tmp["title_len"] = tmp["title"].fillna("").astype(str).str.len()
tmp["text_len"] = tmp["text"].fillna("").astype(str).str.len()
sns.scatterplot(data=tmp.sample(min(4000, len(tmp)), random_state=SEED), x="title_len", y="text_len", s=8, ax=axes[2])
axes[2].set_title("Title vs text length")
plt.tight_layout()
plt.show()

display(label_freq.to_frame())
display(train_df["source"].value_counts().head(20).to_frame("count"))

In [ ]:
train_texts = compose_text_features(train_df)
test_texts = compose_text_features(test_df)

print("Prepared text features")
display(train_texts.head(2))

In [ ]:
# Feature extraction (transformer embeddings with fallback to TF-IDF)
if str(PROJECT_ROOT).startswith("/kaggle/input"):
    ARTIFACTS_DIR = Path("/kaggle/working/artifacts")
else:
    ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_BACKEND = "transformer"
MODEL_NAME = "intfloat/multilingual-e5-small"

try:
    x_train = build_transformer_embeddings(
        train_texts,
        model_name=MODEL_NAME,
        batch_size=32,
        max_length=256,
        cache_path=ARTIFACTS_DIR / "train_embeddings.npz",
    )
    x_test = build_transformer_embeddings(
        test_texts,
        model_name=MODEL_NAME,
        batch_size=32,
        max_length=256,
        cache_path=ARTIFACTS_DIR / "test_embeddings.npz",
    )
except Exception as exc:
    print(f"Transformer path failed: {exc}")
    print("Fallback to TF-IDF features")
    FEATURE_BACKEND = "tfidf"
    x_train, x_test = build_tfidf_features(train_texts, test_texts)

print("Feature backend:", FEATURE_BACKEND)
print("x_train shape:", x_train.shape)
print("x_test shape:", x_test.shape)

In [ ]:
# Ablation block (fast and practical)
# Idea: compare a few model settings on a train subset, then pick best C for final training.

ABLATION_ENABLED = True
ABLATION_ROWS = 6000   # keeps this block fast; set None for full train
ABLATION_SPLITS = 3

BEST_C = 2.0
ablation_df = pd.DataFrame()

if ABLATION_ENABLED:
    if ABLATION_ROWS is None or ABLATION_ROWS >= len(train_df):
        ab_idx = np.arange(len(train_df))
    else:
        ab_idx = train_df.sample(ABLATION_ROWS, random_state=SEED).index.to_numpy()

    y_ab = y[ab_idx]
    x_ab_main = x_train[ab_idx]

    runs = []
    splitter_ab = make_multilabel_splitter(n_splits=ABLATION_SPLITS, seed=SEED)

    for c_value in [1.0, 2.0, 4.0]:
        est = build_base_classifier(seed=SEED, C=c_value)
        cv_ab = fit_cv_models(x_ab_main, y_ab, est, splitter_ab, hamming_score)
        pred_ab = (cv_ab["oof_proba"] >= 0.5).astype(np.int64)
        score_ab = hamming_score(y_ab, pred_ab)
        runs.append({"backend": FEATURE_BACKEND, "C": c_value, "score": score_ab})

    # Optional backend ablation with TF-IDF on same subset.
    try:
        tfidf_train_sub, _ = build_tfidf_features(train_texts.iloc[ab_idx], test_texts.iloc[: min(200, len(test_texts))], max_features_word=40000, max_features_char=25000)
        for c_value in [1.0, 2.0]:
            est = build_base_classifier(seed=SEED, C=c_value)
            cv_ab = fit_cv_models(tfidf_train_sub, y_ab, est, splitter_ab, hamming_score)
            pred_ab = (cv_ab["oof_proba"] >= 0.5).astype(np.int64)
            score_ab = hamming_score(y_ab, pred_ab)
            runs.append({"backend": "tfidf", "C": c_value, "score": score_ab})
    except Exception as exc:
        print(f"TF-IDF ablation skipped: {exc}")

    ablation_df = pd.DataFrame(runs).sort_values("score", ascending=False).reset_index(drop=True)
    display(ablation_df)

    top_main = ablation_df[ablation_df["backend"] == FEATURE_BACKEND]
    if not top_main.empty:
        BEST_C = float(top_main.iloc[0]["C"])

print(f"Selected C for final training: {BEST_C}")

In [ ]:
# CV training
RUN_MODE = "full"  # use "debug" for faster local checks
n_splits = 5 if RUN_MODE == "full" else 2

y = extract_targets(train_df)
splitter = make_multilabel_splitter(n_splits=n_splits, seed=SEED)
estimator = build_base_classifier(seed=SEED, C=BEST_C)

cv_result = fit_cv_models(
    x=x_train,
    y=y,
    estimator=estimator,
    splitter=splitter,
    score_fn=hamming_score,
)

baseline_pred = (cv_result["oof_proba"] >= 0.5).astype(np.int64)
baseline_score = hamming_score(y, baseline_pred)
print(f"OOF hamming_score@0.5: {baseline_score:.6f}")
print("Fold scores:", [round(v, 6) for v in cv_result["fold_scores"]])

In [ ]:
# Threshold tuning
best_thresholds, tuned_score = find_best_thresholds(
    y_true=y,
    y_proba=cv_result["oof_proba"],
    score_fn=hamming_score,
)
print("Best thresholds:", best_thresholds)
print(f"Tuned OOF hamming_score: {tuned_score:.6f}")

In [ ]:
# Train on full data and infer test
full_model = fit_full_model(x_train, y, estimator)
test_proba = full_model.predict_proba(x_test)
test_pred = apply_thresholds(test_proba, best_thresholds)

submission = sample_submission.copy()
submission["id"] = test_df["id"].values
submission["target"] = format_submission_targets(test_pred)

if str(PROJECT_ROOT).startswith("/kaggle/input"):
    output_path = Path("/kaggle/working/sample_submission.csv")
else:
    output_path = PROJECT_ROOT / "sample_submission.csv"

submission.to_csv(output_path, index=False)
print("Saved submission:", output_path)
display(submission.head())

In [ ]:
print("=== Reproducibility report ===")
print("Seed:", SEED)
print("Feature backend:", FEATURE_BACKEND)
print("Selected C:", BEST_C)
print("OOF baseline score:", round(baseline_score, 6))
print("OOF tuned score:", round(tuned_score, 6))
print("Train rows:", len(train_df), "| Test rows:", len(test_df))
if ABLATION_ENABLED and not ablation_df.empty:
    print("Ablation runs:", len(ablation_df))